In [1]:
import json

from typing import List, Dict

from src.tools import search

from langchain_ollama import ChatOllama

from langchain_core.messages import (
    HumanMessage,
    ToolMessage,
    SystemMessage
)

In [2]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [3]:
llm = ChatOllamaretrieve_llm = ChatOllama(
    model="qwen3.5:0.8b",
    reasoning=True,
)

tools = [search]
llm_with_tools = llm.bind_tools(tools, tool_choice=True)

In [4]:
def make_call(call):
    args = call["args"]
    if call["name"] == "search":
        result = search.invoke(args)
    result_json = json.dumps(result, indent=2)
    return ToolMessage(
        content=result_json, 
        tool_call_id=call["id"]
    )

In [5]:
question = "I just discovered the course. Can I join it?"
messages = [
    SystemMessage(content=instructions),
    HumanMessage(content=question)
]

response = llm_with_tools.invoke(messages)

In [6]:
if response.tool_calls:
    tool_calls = response.tool_calls[0]
    print('tool_calls:', tool_calls["name"], tool_calls["args"])
    call_output = make_call(tool_calls)
    messages.append(call_output)

elif response.content:
    print('ASSISTANT:')
    print(response.content)

tool_calls: search {'query': 'join course membership enrollment'}


In [7]:
question = "I just discovered the course. Can I join it?"
messages = [
    SystemMessage(content=instructions),
    HumanMessage(content=question)
]

it = 1
while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = llm_with_tools.invoke(messages)
    messages.append(response)

    if response.tool_calls:
        has_function_calls = True
        
        tool_calls = response.tool_calls[0]
        print('tool_calls:', tool_calls["name"], tool_calls["args"])
        call_output = make_call(tool_calls)
        messages.append(call_output)

    elif response.content:
        print('ASSISTANT:')
        print(response.content)

    it = it + 1
    
    if has_function_calls == False: 
        break

iteration #1...
tool_calls: search {'query': 'join course'}
iteration #2...
tool_calls: search {'query': 'certificate project self-paced mode'}
iteration #3...
tool_calls: search {'query': 'llm-zoomcamp docs logistics'}
iteration #4...
ASSISTANT:
Thank you for your inquiry! I can confirm that you are eligible to join the LLM Zoomcamp course, but to understand how to proceed, let me provide some additional details:

1. **Certificate Requirements**: You'll be eligible to receive a certificate only if you complete the course with a "live" cohort. This means you need to finish the course with peer-review 3 capstone projects after submitting your project.

2. **Project and Self-Paced**: Self-paced learning is not allowed. You can only get a certificate if you finish the course with a "live" cohort. The certificate process requires peer-review of 3 capstone projects that you submit after submitting your project and the form is closed.

3. **Course Start Timing**: You can start whenever you w

In [8]:
def agent_loop(instructions, question, llm_w_tools) -> str:
    messages = [
        SystemMessage(content=instructions),
        HumanMessage(content=question)
    ]

    it = 1
    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = llm_w_tools.invoke(messages)
        messages.append(response)

        if response.tool_calls:
            has_function_calls = True
            
            tool_calls = response.tool_calls[0]
            print('tool_calls:', tool_calls["name"], tool_calls["args"])
            call_output = make_call(tool_calls)
            messages.append(call_output)

        elif response.content:
            print('ASSISTANT:')
            last_answer = response.content
            print(response.content)

        it = it + 1
        
        if has_function_calls == False: 
            break

    return last_answer

In [10]:
agent_loop(instructions, "How do I run Olama locally?", llm_with_tools)

iteration #1...
tool_calls: search {'query': 'olama'}
iteration #2...
tool_calls: search {'query': 'olama running'}
iteration #3...
tool_calls: search {'query': 'olama database'}
iteration #4...
tool_calls: search {'query': 'olama course'}
iteration #5...
tool_calls: search {'query': 'olama database'}
iteration #6...
tool_calls: search {'query': 'olama python'}
iteration #7...
tool_calls: search {'query': 'olama docker'}
iteration #8...
tool_calls: search {'query': 'olama SQL'}
iteration #9...
tool_calls: search {'query': 'data ingestion Python'}
iteration #10...
tool_calls: search {'query': 'ollama database'}
iteration #11...
tool_calls: search {'query': 'python sql database'}
iteration #12...
tool_calls: search {'query': 'docker docker-compose'}
iteration #13...
tool_calls: search {'query': 'PostgreSQL connection PostgreSQL database host name'}
iteration #14...
ASSISTANT:
# Module 5: Monitoring

## 1. Python Import Error
```
OSError: [WinError 126] The specified module could not be f

'# Module 5: Monitoring\n\n## 1. Python Import Error\n```\nOSError: [WinError 126] The specified module could not be found. Error loading "C:\\Users\\USER\\AppData\\Local\\Programs\\Python\\Python310\\lib\\site-packages\\torch\\lib\\fbgemm.dll"\n```\n\n**Solution:** \n1. Install **Visual C++ Redistributable** for Windows (not Visual Studio Code)\n2. Ensure all dependencies are properly installed\n\n**Source:** [discuss.pytorch.org](https://discuss.pytorch.org/t/failed-to-import-pytorch-fbgemm-dll-or-one-of-its-dependencies-is-missing/201969)\n\n---\n\n## 2. Docker Compose Application Error\n```\nError: failed to create task for container: failed to create shim task: OCI runtime create failed: runc create failed: unable to start container process: exec: "streamlit": executable file not found in $PATH\n```\n\n**Solution:**\n1. Create a Dockerfile for the Streamlit app\n2. Add `streamlit` to your `docker-compose` definition\n3. Run to rebuild:\n```bash\ndocker-compose up --build\n```\n\n-

In [11]:
agent_loop(instructions, "I just discovered the course. Can I still join it?", llm_with_tools)

iteration #1...
tool_calls: search {'query': 'course registration available now'}
iteration #2...
tool_calls: search {'query': 'enrollment application join course'}
iteration #3...
tool_calls: search {'query': 'apply enroll join application course data talks'}
iteration #4...
ASSISTANT:
Based on the information I found, **yes, you can still join the course**.

The key details are:

- **Current status**: Yes, you're accepted. You can start learning and submitting homework without registration.
- **Certificate requirement**: If you want to receive a certificate, you need to submit your project while the course is still accepting submissions.
- **When offered**: The course is offered for the Summer 2027 term.
- **Workflow**: You can start when you want - just watch the lesson videos, work through the lesson notebooks/code, read the homework instructions on GitHub, and submit answers through the platform before the deadline.

I'm also glad to find that there were no separate live sessions 

"Based on the information I found, **yes, you can still join the course**.\n\nThe key details are:\n\n- **Current status**: Yes, you're accepted. You can start learning and submitting homework without registration.\n- **Certificate requirement**: If you want to receive a certificate, you need to submit your project while the course is still accepting submissions.\n- **When offered**: The course is offered for the Summer 2027 term.\n- **Workflow**: You can start when you want - just watch the lesson videos, work through the lesson notebooks/code, read the homework instructions on GitHub, and submit answers through the platform before the deadline.\n\nI'm also glad to find that there were no separate live sessions for each module, as the materials are pre-recorded and available.\n\nIs there any other area you'd like to explore about this course?"

In [12]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?", llm_with_tools)

iteration #1...
tool_calls: search {'query': 'course join enrollment registration process'}
iteration #2...
tool_calls: search {'query': 'llm-zoomcamp course details structure'}
iteration #3...
tool_calls: search {'query': 'enrollment confirmation register submit project'}
iteration #4...
ASSISTANT:
Based on my research, yes, you can join the course! 

According to the course information:

1. **Course Status**: The course is accepted, you're already enrolled, and you can actually start learning without needing to register for enrollment.

2. **Registration Purpose**: Registration is primarily a tool to gauge interest before the course starts, but the course accepts all students.

3. **How to Track Progress**: To monitor your progress through the course, you'll need to use the **LLM Zoomcamp course management platform** at https://courses.datatalks.club/llm-zoomcamp-2026/.

4. **Projects**: You only need to submit one project (not two), and the submission allows for improvement during s

"Based on my research, yes, you can join the course! \n\nAccording to the course information:\n\n1. **Course Status**: The course is accepted, you're already enrolled, and you can actually start learning without needing to register for enrollment.\n\n2. **Registration Purpose**: Registration is primarily a tool to gauge interest before the course starts, but the course accepts all students.\n\n3. **How to Track Progress**: To monitor your progress through the course, you'll need to use the **LLM Zoomcamp course management platform** at https://courses.datatalks.club/llm-zoomcamp-2026/.\n\n4. **Projects**: You only need to submit one project (not two), and the submission allows for improvement during subsequent attempts if you don't meet the deadline for attempt 1.\n\n5. **Certification Process**: To earn a certificate, you need to complete 3 peer-review sessions. Once your first attempt submission is made, you can review previous project submissions.\n\n6. **Support**: \n   - Telegram 

In [13]:
agent_loop(instructions, "what's queen gambit?", llm_with_tools)

iteration #1...
tool_calls: search {'query': 'queen gambit chess'}
iteration #2...
tool_calls: search {'query': 'queen gambit game theory chess'}
iteration #3...
tool_calls: search {'query': 'queen gambit chess theory'}
iteration #4...
tool_calls: search {'query': 'Queen Gambit title'}
iteration #5...
tool_calls: search {'query': 'Gambit chess'}
iteration #6...
tool_calls: search {'query': 'Queen Bishop Gambit'}
iteration #7...
tool_calls: search {'query': 'Queen Gambit'}
iteration #8...
tool_calls: search {'query': 'opening chess'}
iteration #9...
tool_calls: search {'query': 'move chess'}
iteration #10...
tool_calls: search {'query': 'Gambit opening'}
iteration #11...
tool_calls: search {'query': 'Queen Gambit move'}
iteration #12...
tool_calls: search {'query': 'queen gambit'}
iteration #13...
tool_calls: search {'query': 'Gambit'}
iteration #14...
tool_calls: search {'query': 'Gambit 1922'}
iteration #15...
tool_calls: search {'query': 'Queen Gambit of'}
iteration #16...
tool_calls

'I apologize, but I don\'t have specific information about "Gambit chess" or "Queen Gambit" in my knowledge base. These are typically associated with specific chess openings or strategies, and if you\'re asking about a different course (like LLM Zoomcamp), feel free to clarify which course you\'d like help with!\n\nIf you have questions about:\n- LLM Zoomcamp course topics and workflow\n- LLM Zoomcamp course structure and logistics\n- Other DataTalks.Club courses or courses\n- General Discord-related questions\n\nPlease let me know, and I\'d be happy to help you!'

In [14]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?", llm_with_tools)

iteration #1...
tool_calls: search {'query': 'queen gambit'}
iteration #2...
tool_calls: search {'query': 'Queen Gambit strategy'}
iteration #3...
tool_calls: search {'query': 'Queen Gambit move'}
iteration #4...
tool_calls: search {'query': 'Queen Gambit position in chess'}
iteration #5...
tool_calls: search {'query': 'Gambit queen'}
iteration #6...
tool_calls: search {'query': 'Queen Gambit theory'}
iteration #7...
tool_calls: search {'query': 'Queen Gambit chess'}
iteration #8...
ASSISTANT:
I've searched the course FAQ database many times for information about "Queen Gambit" and found nothing relevant. This term appears to be related to chess strategy or position concepts, but the FAQ database does not contain official entries for this topic.

Based on my search results so far, there's no information in the FAQ database about Queen Gambit. The course FAQ is for course and logistics-related questions, not chess strategy or position details.

I'd be happy to help with other questions 

'I\'ve searched the course FAQ database many times for information about "Queen Gambit" and found nothing relevant. This term appears to be related to chess strategy or position concepts, but the FAQ database does not contain official entries for this topic.\n\nBased on my search results so far, there\'s no information in the FAQ database about Queen Gambit. The course FAQ is for course and logistics-related questions, not chess strategy or position details.\n\nI\'d be happy to help with other questions from the course or look up information on topics that might appear in the FAQ database. Could you explore any other areas you\'d like to discuss?'